# MLP - مشكلة XOR - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## لماذا XOR؟
مشكلة XOR هي المثال الكلاسيكي الذي أثبت أن **Perceptron بطبقة واحدة** لا يستطيع حل مشاكل غير قابلة للفصل خطياً.
جدول الحقيقة: (0,0)→0، (0,1)→1، (1,0)→1، (1,1)→0. لا يمكن رسم خط واحد يفصل الفئتين.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز X و y
4. تقسيم البيانات
5. Feature scaling (Standardization)
6. Perceptron خطي (طبقة واحدة) — baseline يفشل تقريباً
7. MLP (طبقة مخفية + ReLU) — يحل XOR
8. مقارنة الدقة
9. رسم حدود القرار


## الخطوة 1: استيراد المكتبات

- **numpy**: حسابات رقمية وبناء شبكة الشبكة للتنبؤ على الشبكة
- **matplotlib**: رسم حدود القرار
- **pandas**: قراءة ملف CSV
- **sklearn**: تقسيم البيانات وStandardScaler
- **tensorflow.keras**: بناء وتدريب الشبكات العصبية


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:

- **`numpy` (المستوردة كـ `np`):** للعمليات الحسابية والتعامل مع المصفوفات الرياضية.
- **`pandas` (المستوردة كـ `pd`):** لقراءة البيانات وإدارة الجداول البرمجية (DataFrames).
- **`matplotlib.pyplot` (المستوردة كـ `plt`):** للرسم البياني وتصور البيانات بصرياً.
- **`train_test_split`:** لتقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار بشكل عشوائي ومنظم.
- **`StandardScaler`:** لتقييس وتوحيد نطاق الخصائص (Feature Scaling) ليكون المتوسط صفر والانحراف المعياري واحد.
- **`tensorflow / keras`:** لبناء وتدريب الشبكات العصبية الاصطناعية ونماذج التعلم العميق.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q  # فعّل هذا السطر في Google Colab إذا لزم
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


## الخطوة 2: قراءة البيانات

الملف `XOR.csv` يحتوي على نقطتين (x1, x2) وتسمية (label).
البيانات موزعة حول زوايا مربع XOR مع ضوضاء بسيطة لمحاكاة واقعية أكثر.


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:




In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'XOR.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/1-%20MLP%20XOR/XOR.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: تجهيز X و y

- **X**: الميزات (x1, x2)
- **y**: التسمية الثنائية (0 أو 1)


### خطوة: الخطوة 3) تجهيز X و y
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 3) تجهيز X و y
X = dataset[['x1', 'x2']].values
y = dataset['label'].values


## الخطوة 4: تقسيم البيانات

نستخدم 80% للتدريب و20% للاختبار. `random_state=0` لإعادة نفس النتائج في كل تشغيل.


### ثامناً: تقسيم البيانات إلى مجموعتي تدريب واختبار (Train/Test Split)
نقسم البيانات بنسبة 20% لمجموعة الاختبار وبقية البيانات لمجموعة التدريب:
- **بيانات التدريب (Training Set):** لتعليم النموذج وضبط أوزانه ومعاملاته.
- **بيانات الاختبار (Test Set):** لتقييم النموذج واختبار قدرته على التنبؤ ببيانات جديدة كلياً.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)


## الخطوة 5: Feature Scaling

StandardScaler يحوّل كل ميزة لمتوسط 0 وانحراف معياري 1.
**لماذا؟** الشبكات العصبية تتعلم أسرع عندما تكون الميزات على نفس المقياس.
نطبّق `fit` على التدريب فقط ثم `transform` على الاختبار لتجنب تسرب البيانات.


### تاسعاً: تقييس الخصائص (Feature Scaling)
نقوم بعملية التقييس أو المعايرة للبيانات:
- نستخدم `fit_transform` على مجموعة التدريب ليتعلم المتوسط والانحراف المعياري ويطبق التحويل.
- نستخدم `transform` فقط على مجموعة الاختبار لمنع تسرب البيانات (Data Leakage).
- هذه الخطوة ضرورية جداً للخوارزميات الحساسة للمقاييس مثل متجهات الدعم (SVM)، الجار الأقرب (KNN)، والشبكات العصبية.


In [ ]:
# الخطوة 5) Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


## الخطوة 6: Perceptron خطي (Baseline)

نموذج ب**طبقة واحدة** + **sigmoid** = Perceptron كلاسيكي.

**ما نتوقعه للطلاب:** دقة قريبة من 50–60% لأن XOR غير قابل للفصل خطياً.
لا يمكن لخط واحد (أو hyperplane) فصل الفئتين في XOR.


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 6) Perceptron خطي - طبقة واحدة
perceptron = Sequential([
    Dense(units=1, activation='sigmoid', input_shape=(2,))
])
perceptron.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
perceptron.fit(X_train, y_train, batch_size=16, epochs=100, verbose=0)
perceptron_loss, perceptron_acc = perceptron.evaluate(X_test, y_test, verbose=0)
print(f'Perceptron test accuracy: {perceptron_acc:.2f}')


## الخطوة 7: MLP (شبكة متعددة الطبقات)

نضيف **طبقة مخفية** (8 neurons) مع **ReLU** ثم طبقة إخراج sigmoid.

**ReLU** يسمح بتمثيلات غير خطية؛ الطبقة المخفية تبني features مركبة تفصل XOR.
**ما نتوقعه:** دقة قريبة من 100% على الاختبار.


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 7) MLP - طبقة مخفية + ReLU
mlp = Sequential([
    Dense(units=8, activation='relu', input_shape=(2,)),
    Dense(units=1, activation='sigmoid')
])
mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
mlp.fit(X_train, y_train, batch_size=16, epochs=100, verbose=0)
mlp_loss, mlp_acc = mlp.evaluate(X_test, y_test, verbose=0)
print(f'MLP test accuracy: {mlp_acc:.2f}')


## الخطوة 8: مقارنة النتائج

اطلب من الطلاب ملاحظة الفجوة الكبيرة بين Perceptron و MLP.
هذا هو جوهر **Universal Approximation**: طبقة مخفية واحدة كافية لـ XOR.


### خطوة: الخطوة 8) مقارنة الدقة
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 8) مقارنة الدقة
print('--- نتائج XOR ---')
print(f'Perceptron خطي: {perceptron_acc:.2%}')
print(f'MLP (طبقة مخفية): {mlp_acc:.2%}')
print('XOR غير قابل للفصل خطياً — لذلك MLP يتفوق على طبقة واحدة.')


## الخطوة 9: رسم حدود القرار

1. نبني شبكة نقاط على مستوى (x1, x2)
2. نتنبأ باحتمال الفئة 1 لكل نقطة
3. `contourf` يرسم المناطق (أزرق/أحمر)
4. النقاط الزرقاء = Class 0، الحمراء = Class 1

**للمحاضر:** Perceptron يظهر حداً خطياً تقريباً؛ MLP يرسم حداً منحنياً يفصل XOR.


### الرابع عشر: رسم النتائج بيانيا (Visualization of Results)
نقوم برسم النقاط الحقيقية (الحمراء عادة) والخط أو المنحنى الممثل للنموذج (الأزرق) بصرياً:
- يساعدنا الرسم البياني في التحقق البصري المباشر من مدى دقة التوقعات وملاءمة النموذج للبيانات الحقيقية.


In [ ]:
# الخطوة 9) رسم حدود القرار
def plot_decision_boundary(model, title, X_scaled, y_raw, scaler):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    Z = model.predict(grid_scaled, verbose=0).reshape(xx.shape)

    plt.figure(figsize=(5, 4))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X[y_raw == 0, 0], X[y_raw == 0, 1], c='blue', edgecolors='k', label='Class 0')
    plt.scatter(X[y_raw == 1, 0], X[y_raw == 1, 1], c='red', edgecolors='k', label='Class 1')
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.legend()
    plt.show()

plot_decision_boundary(perceptron, 'Linear Perceptron', X_train, y, sc)
plot_decision_boundary(mlp, 'MLP with Hidden Layer', X_train, y, sc)
